# AlpCAN CXR Pipeline: Multi-Model Ensemble

**Amaç:** 3 farklı CXR modelinin (TorchXRayVision DenseNet-121, ResNet-50, Ark+ Swin-L) tahminlerini birleştirerek ensemble performansını değerlendirmek.

**İçindekiler:**
1. Kurulum ve Konfigürasyon
2. Tahminlerin Yüklenmesi
3. Bireysel Model Performansları
4. Ensemble Yöntemleri (Basit Ortalama, Ağırlıklı, Maksimum, Rank, Optimal)
5. Ensemble Sonuçları Karşılaştırması
6. Patoloji Bazında Derinlemesine Analiz
7. Güven Analizi (Tahmin Uyumu)
8. AlpCAN Pipeline Entegrasyonu
9. Görselleştirme Özeti

**Modeller:**
- **Notebook 02:** TorchXRayVision DenseNet-121 (Ortalama AUC ~0.796)
- **Notebook 03:** ResNet-50 512px (Ortalama AUC ~0.830)
- **Notebook 05:** Ark+ Swin-L 768px (Ortalama AUC ~0.878)

**Not:** Bu notebook GPU gerektirmez — sadece CSV tahminler üzerinde çalışır.

**Referanslar:**
- Wang et al., CVPR 2017 (ChestX-ray14)
- Haghighi et al., IEEE TMI 2022 (Foundation Ark)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import product
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata

# Gorsellestrme ayarlari
sns.set_style('darkgrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Seaborn: {sns.__version__}")
print("GPU: Gerekli degil (CPU-only notebook)")

In [ ]:
# === Konfiguerasyon ===
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model tahmin dosyalari
PRED_PATHS = {
    'TorchXRayVision': INPUT_DIR / 'alpcan-cxr-pipeline-torchxrayvision-baseline' / 'cxr_predictions_full.csv',
    'ResNet-50': INPUT_DIR / 'alpcan-cxr-pipeline-resnet-50-512-baseline' / 'cxr_resnet50_predictions_full.csv',
    'Ark+': INPUT_DIR / 'alpcan-cxr-pipeline-ark-zero-shot-baseline' / 'cxr_ark_predictions_full.csv',
}

# Ortak 14 NIH patolojisi (uc modelin kesisimi)
COMMON_14 = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

# Model isimleri (kisa)
MODEL_NAMES = list(PRED_PATHS.keys())

# Renk paleti
MODEL_COLORS = {
    'TorchXRayVision': '#3498db',
    'ResNet-50': '#2ecc71',
    'Ark+': '#e74c3c',
}

ENSEMBLE_COLORS = {
    'Simple Average': '#9b59b6',
    'Weighted Average': '#e67e22',
    'Maximum': '#1abc9c',
    'Rank Average': '#34495e',
    'Optimal Weighted': '#f1c40f',
}

print("Konfiguerasyon tamamlandi.")
print(f"Ortak patoloji sayisi: {len(COMMON_14)}")
print(f"Model sayisi: {len(MODEL_NAMES)}")
for name, path in PRED_PATHS.items():
    exists = path.exists()
    status = 'MEVCUT' if exists else 'BULUNAMADI'
    print(f"  {name}: {status} -- {path}")

---
## 2. Tahminlerin Yüklenmesi

In [ ]:
# Her modelin tahmin CSV'sini yukle
raw_dfs = {}
for name, path in PRED_PATHS.items():
    if path.exists():
        df = pd.read_csv(path)
        raw_dfs[name] = df
        print(f"{name}:")
        print(f"  Dosya: {path.name}")
        print(f"  Satir: {len(df):,}")
        print(f"  Sutun: {df.shape[1]}")
        pred_cols = [c for c in df.columns if c.startswith('pred_')]
        gt_cols = [c for c in df.columns if c.startswith('gt_')]
        print(f"  Tahmin sutunlari ({len(pred_cols)}): {pred_cols[:5]}...")
        print(f"  GT sutunlari ({len(gt_cols)}): {gt_cols[:5]}...")
        print()
    else:
        print(f"{name}: DOSYA BULUNAMADI - {path}")
        print(f"  kernel_sources'a ilgili notebook eklenmis mi kontrol edin.")
        print()

assert len(raw_dfs) == 3, f"3 model bekleniyor, {len(raw_dfs)} yuklendi!"

In [ ]:
def normalize_column_name(col):
    """Sutun isimlerini normalize et: 'pred_Lung Lesion' -> 'pred_Lung_Lesion'"""
    return col.replace(' ', '_')


def extract_predictions(df, model_name, pathologies):
    """Bir model DataFrame'inden belirtilen patolojilerin tahmin ve GT degerlerini cikar.
    
    Returns:
        pred_df: image_name + pred sutunlari
        gt_df: image_name + gt sutunlari
    """
    # Sutun isimlerini normalize et
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    
    result = {'image_name': df['image_name'].values}
    gt_result = {'image_name': df['image_name'].values}
    
    missing_pred = []
    missing_gt = []
    
    for p in pathologies:
        pred_col = f'pred_{p}'
        gt_col = f'gt_{p}'
        
        if pred_col in df.columns:
            result[p] = df[pred_col].values
        else:
            missing_pred.append(p)
            
        if gt_col in df.columns:
            gt_result[p] = df[gt_col].values
        else:
            missing_gt.append(p)
    
    if missing_pred:
        print(f"  UYARI [{model_name}]: Tahmin sutunu bulunamadi: {missing_pred}")
    if missing_gt:
        print(f"  UYARI [{model_name}]: GT sutunu bulunamadi: {missing_gt}")
    
    pred_df = pd.DataFrame(result)
    gt_df = pd.DataFrame(gt_result)
    
    return pred_df, gt_df


# Her model icin tahmin ve GT cikar
model_preds = {}
model_gts = {}

for name, df in raw_dfs.items():
    print(f"\n{name} isleniyor...")
    pred_df, gt_df = extract_predictions(df, name, COMMON_14)
    model_preds[name] = pred_df
    model_gts[name] = gt_df
    print(f"  Tahmin shape: {pred_df.shape}")
    print(f"  GT shape: {gt_df.shape}")

print("\nTum modeller islendi.")

In [ ]:
# Ortak goruntuler uzerinden hizalama
# Tum modellerin kesisimindeki goruntuleri bul
image_sets = {name: set(pred['image_name'].values) for name, pred in model_preds.items()}

print("Model bazinda goruntu sayilari:")
for name, img_set in image_sets.items():
    print(f"  {name}: {len(img_set):,}")

# Kesisim
common_images = sorted(set.intersection(*image_sets.values()))
print(f"\nOrtak goruntu sayisi (kesisim): {len(common_images):,}")

# Hizalanmis DataFrames
aligned_preds = {}
aligned_gt = None

for name in MODEL_NAMES:
    pred_df = model_preds[name]
    gt_df = model_gts[name]
    
    # Ortak goruntulerle filtrele ve sirala
    pred_aligned = pred_df[pred_df['image_name'].isin(common_images)].copy()
    pred_aligned = pred_aligned.sort_values('image_name').reset_index(drop=True)
    aligned_preds[name] = pred_aligned
    
    # GT sadece bir kez alinir (tum modellerde ayni olmali)
    if aligned_gt is None:
        gt_aligned = gt_df[gt_df['image_name'].isin(common_images)].copy()
        gt_aligned = gt_aligned.sort_values('image_name').reset_index(drop=True)
        aligned_gt = gt_aligned

# Dogrulama: tum image_name'ler ayni mi?
ref_names = aligned_preds[MODEL_NAMES[0]]['image_name'].values
for name in MODEL_NAMES[1:]:
    assert np.array_equal(ref_names, aligned_preds[name]['image_name'].values), \
        f"{name} hizalama hatasi!"
assert np.array_equal(ref_names, aligned_gt['image_name'].values), "GT hizalama hatasi!"

N_IMAGES = len(common_images)
print(f"\nHizalama tamamlandi: {N_IMAGES:,} goruntu x {len(COMMON_14)} patoloji x {len(MODEL_NAMES)} model")

# Patoloji bazinda pozitif sayilari
print(f"\nPatoloji dagilimi (N={N_IMAGES:,}):")
for p in COMMON_14:
    if p in aligned_gt.columns:
        n_pos = int(aligned_gt[p].sum())
        pct = n_pos / N_IMAGES * 100
        marker = " << AlpCAN" if p in ('Nodule', 'Mass') else ""
        print(f"  {p:>20s}: {n_pos:>6,} ({pct:>5.1f}%){marker}")

---
## 3. Bireysel Model Performanslari

In [ ]:
# Her model icin patoloji bazinda AUC hesapla
individual_aucs = {}  # {model_name: {pathology: auc}}

print("=" * 80)
print("Bireysel Model Performanslari (AUC-ROC)")
print(f"N = {N_IMAGES:,} goruntu")
print("=" * 80)

header = f"{'Patoloji':>20s}"
for name in MODEL_NAMES:
    header += f" | {name:>16s}"
header += " | En Iyi Model"
print(header)
print("-" * 100)

for name in MODEL_NAMES:
    individual_aucs[name] = {}

for p in COMMON_14:
    y_true = aligned_gt[p].values
    n_pos = int(y_true.sum())
    
    if n_pos == 0 or n_pos == N_IMAGES:
        line = f"  {p:>20s}"
        for name in MODEL_NAMES:
            line += f" | {'N/A':>16s}"
            individual_aucs[name][p] = np.nan
        print(line)
        continue
    
    line = f"  {p:>20s}"
    best_auc = -1
    best_model = ""
    
    for name in MODEL_NAMES:
        y_score = aligned_preds[name][p].values
        auc = roc_auc_score(y_true, y_score)
        individual_aucs[name][p] = auc
        line += f" | {auc:>16.4f}"
        if auc > best_auc:
            best_auc = auc
            best_model = name
    
    marker = " <<" if p in ('Nodule', 'Mass') else ""
    line += f" | {best_model}{marker}"
    print(line)

print("-" * 100)

# Ortalama AUC
line = f"  {'ORTALAMA':>20s}"
for name in MODEL_NAMES:
    valid_aucs = [v for v in individual_aucs[name].values() if not np.isnan(v)]
    mean_auc = np.mean(valid_aucs)
    line += f" | {mean_auc:>16.4f}"
print(line)

# Nodule + Mass ortalama
line = f"  {'Nodule+Mass ort.':>20s}"
for name in MODEL_NAMES:
    cancer_auc = np.mean([individual_aucs[name].get('Nodule', np.nan),
                          individual_aucs[name].get('Mass', np.nan)])
    line += f" | {cancer_auc:>16.4f}"
line += " | <- AlpCAN odak"
print(line)

In [ ]:
# Karsilastirma tablosu DataFrame
auc_matrix = pd.DataFrame(individual_aucs).reindex(COMMON_14)
auc_matrix.index.name = 'Patoloji'

print("AUC Matrisi:")
print(auc_matrix.round(4).to_string())

# Hangi model hangi patolojide en iyi?
best_per_pathology = auc_matrix.idxmax(axis=1)
print(f"\nPatoloji bazinda en iyi model:")
for p, best in best_per_pathology.items():
    auc_val = auc_matrix.loc[p, best]
    print(f"  {p:>20s}: {best} ({auc_val:.4f})")

print(f"\nModel bazinda 'en iyi' patoloji sayisi:")
for name in MODEL_NAMES:
    count = (best_per_pathology == name).sum()
    print(f"  {name}: {count}/14 patolojide en iyi")

In [ ]:
# Isitma haritasi (heatmap) gorsellistirmesi
fig, ax = plt.subplots(figsize=(10, 9))

heatmap_data = auc_matrix.copy()

sns.heatmap(
    heatmap_data,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    vmin=0.60, vmax=1.0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'AUC-ROC'}
)

ax.set_title('Bireysel Model AUC-ROC Performansi\n(14 Ortak NIH Patolojisi)', fontweight='bold')
ax.set_ylabel('Patoloji')
ax.set_xlabel('Model')

# Nodule ve Mass satirlarini vurgula
for i, p in enumerate(COMMON_14):
    if p in ('Nodule', 'Mass'):
        ax.add_patch(plt.Rectangle((0, i), len(MODEL_NAMES), 1,
                                    fill=False, edgecolor='red', linewidth=2.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_heatmap_individual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_heatmap_individual.png")

---
## 4. Ensemble Yöntemleri

In [ ]:
# Yardimci fonksiyonlar
def get_pred_array(model_name, pathology):
    """Belirtilen model ve patoloji icin tahmin dizisini dondur."""
    return aligned_preds[model_name][pathology].values


def get_all_preds(pathology):
    """3 modelin tahminlerini (N, 3) array olarak dondur."""
    arrays = [get_pred_array(name, pathology) for name in MODEL_NAMES]
    return np.column_stack(arrays)


def compute_ensemble_auc(ensemble_preds, pathology):
    """Ensemble tahmini icin AUC hesapla."""
    y_true = aligned_gt[pathology].values
    n_pos = int(y_true.sum())
    if n_pos == 0 or n_pos == len(y_true):
        return np.nan
    return roc_auc_score(y_true, ensemble_preds)


print("Yardimci fonksiyonlar tanimlandi.")
print(f"Ornek: get_all_preds('Nodule').shape = {get_all_preds('Nodule').shape}")

In [ ]:
# === ENSEMBLE 1: Basit Ortalama ===
# Uc modelin olasilik degerlerinin aritmetik ortalamasi
simple_avg_aucs = {}
simple_avg_preds = {}  # patoloji -> tahmin dizisi

print("Ensemble 1: Basit Ortalama")
print("-" * 50)

for p in COMMON_14:
    all_p = get_all_preds(p)  # (N, 3)
    ensemble_pred = all_p.mean(axis=1)  # (N,)
    simple_avg_preds[p] = ensemble_pred
    auc = compute_ensemble_auc(ensemble_pred, p)
    simple_avg_aucs[p] = auc
    if not np.isnan(auc):
        best_ind = max(individual_aucs[m][p] for m in MODEL_NAMES)
        diff = auc - best_ind
        print(f"  {p:>20s}: AUC={auc:.4f} (vs en iyi bireysel: {diff:+.4f})")

valid_aucs = [v for v in simple_avg_aucs.values() if not np.isnan(v)]
print(f"\n  Ortalama AUC: {np.mean(valid_aucs):.4f}")

In [ ]:
# === ENSEMBLE 2: Agirlikli Ortalama ===
# Agirliklar bireysel model ortalama AUC'sine oranli
model_mean_aucs = {}
for name in MODEL_NAMES:
    valid = [v for v in individual_aucs[name].values() if not np.isnan(v)]
    model_mean_aucs[name] = np.mean(valid)

# AUC'ye oranli agirliklar
total_auc = sum(model_mean_aucs.values())
global_weights = {name: auc_val / total_auc for name, auc_val in model_mean_aucs.items()}

print("Ensemble 2: Agirlikli Ortalama")
print(f"  Agirliklar (genel AUC'ye oranli):")
for name, w in global_weights.items():
    print(f"    {name}: {w:.4f} (AUC={model_mean_aucs[name]:.4f})")

weighted_avg_aucs = {}
weighted_avg_preds = {}

print(f"\n{'':>20s} | {'Agirlikli':>10s} | {'Basit':>10s} | {'Fark':>8s}")
print("-" * 60)

for p in COMMON_14:
    all_p = get_all_preds(p)  # (N, 3)
    w_array = np.array([global_weights[name] for name in MODEL_NAMES])
    ensemble_pred = np.average(all_p, axis=1, weights=w_array)
    weighted_avg_preds[p] = ensemble_pred
    auc = compute_ensemble_auc(ensemble_pred, p)
    weighted_avg_aucs[p] = auc
    if not np.isnan(auc):
        diff = auc - simple_avg_aucs[p]
        print(f"  {p:>20s} | {auc:>10.4f} | {simple_avg_aucs[p]:>10.4f} | {diff:>+7.4f}")

valid_aucs = [v for v in weighted_avg_aucs.values() if not np.isnan(v)]
print(f"\n  Ortalama AUC: {np.mean(valid_aucs):.4f}")

In [ ]:
# === ENSEMBLE 3: Maksimum ===
# Her goruntu icin 3 model arasindaki en yuksek olasilik degeri
max_aucs = {}
max_preds = {}

print("Ensemble 3: Maksimum")
print("-" * 50)

for p in COMMON_14:
    all_p = get_all_preds(p)  # (N, 3)
    ensemble_pred = all_p.max(axis=1)  # (N,)
    max_preds[p] = ensemble_pred
    auc = compute_ensemble_auc(ensemble_pred, p)
    max_aucs[p] = auc
    if not np.isnan(auc):
        best_ind = max(individual_aucs[m][p] for m in MODEL_NAMES)
        diff = auc - best_ind
        print(f"  {p:>20s}: AUC={auc:.4f} (vs en iyi bireysel: {diff:+.4f})")

valid_aucs = [v for v in max_aucs.values() if not np.isnan(v)]
print(f"\n  Ortalama AUC: {np.mean(valid_aucs):.4f}")

In [ ]:
# === ENSEMBLE 4: Rank Ortalamasi ===
# Her modelin tahminlerini percentile'a cevir, sonra ortalama al
rank_avg_aucs = {}
rank_avg_preds = {}

print("Ensemble 4: Rank (Percentile) Ortalamasi")
print("-" * 50)

for p in COMMON_14:
    all_p = get_all_preds(p)  # (N, 3)
    
    # Her modelin tahminlerini percentile'a cevir
    ranked = np.zeros_like(all_p)
    for j in range(all_p.shape[1]):
        ranked[:, j] = rankdata(all_p[:, j]) / len(all_p[:, j])
    
    ensemble_pred = ranked.mean(axis=1)  # (N,)
    rank_avg_preds[p] = ensemble_pred
    auc = compute_ensemble_auc(ensemble_pred, p)
    rank_avg_aucs[p] = auc
    if not np.isnan(auc):
        best_ind = max(individual_aucs[m][p] for m in MODEL_NAMES)
        diff = auc - best_ind
        print(f"  {p:>20s}: AUC={auc:.4f} (vs en iyi bireysel: {diff:+.4f})")

valid_aucs = [v for v in rank_avg_aucs.values() if not np.isnan(v)]
print(f"\n  Ortalama AUC: {np.mean(valid_aucs):.4f}")

In [ ]:
# === ENSEMBLE 5: Optimal Agirlikli (Oracle) ===
# Her patoloji icin ayri agirlik grid search
# NOT: Bu bir oracle analizidir — tam veri uzerinde optimize edilir!
# Gercek uygulamada cross-validation ile yapilmalidir.

optimal_aucs = {}
optimal_preds = {}
optimal_weights_per_pathology = {}  # JSON'a kaydedilecek

print("Ensemble 5: Optimal Agirlikli (Oracle Grid Search)")
print("UYARI: Bu bir oracle analizidir! Gercek performans icin cross-validation gereklidir.")
print("-" * 70)

# Grid: w1, w2 = 0..1 (0.05 adim), w3 = 1 - w1 - w2
weight_grid = np.arange(0.0, 1.05, 0.05)

for p in COMMON_14:
    y_true = aligned_gt[p].values
    n_pos = int(y_true.sum())
    
    if n_pos == 0 or n_pos == len(y_true):
        optimal_aucs[p] = np.nan
        continue
    
    all_p = get_all_preds(p)  # (N, 3)
    
    best_auc = -1
    best_w = (1/3, 1/3, 1/3)
    
    for w1 in weight_grid:
        for w2 in weight_grid:
            w3 = 1.0 - w1 - w2
            if w3 < -0.01 or w3 > 1.01:
                continue
            w3 = max(0.0, w3)
            
            weights = np.array([w1, w2, w3])
            ensemble_pred = np.average(all_p, axis=1, weights=weights)
            auc = roc_auc_score(y_true, ensemble_pred)
            
            if auc > best_auc:
                best_auc = auc
                best_w = (w1, w2, w3)
    
    optimal_aucs[p] = best_auc
    optimal_weights_per_pathology[p] = {
        MODEL_NAMES[0]: round(best_w[0], 2),
        MODEL_NAMES[1]: round(best_w[1], 2),
        MODEL_NAMES[2]: round(best_w[2], 2),
    }
    
    # Optimal agirliklarla tahmin olustur
    w_arr = np.array(best_w)
    optimal_preds[p] = np.average(all_p, axis=1, weights=w_arr)
    
    best_ind = max(individual_aucs[m][p] for m in MODEL_NAMES)
    diff = best_auc - best_ind
    w_str = f"w=[{best_w[0]:.2f}, {best_w[1]:.2f}, {best_w[2]:.2f}]"
    print(f"  {p:>20s}: AUC={best_auc:.4f} (vs en iyi bireysel: {diff:+.4f}) {w_str}")

valid_aucs = [v for v in optimal_aucs.values() if not np.isnan(v)]
print(f"\n  Ortalama AUC: {np.mean(valid_aucs):.4f}")

# Agirlik ozeti
print(f"\nOptimal Agirlik Ozeti:")
for p, w_dict in optimal_weights_per_pathology.items():
    dominant = max(w_dict, key=w_dict.get)
    print(f"  {p:>20s}: {w_dict} -> Dominant: {dominant}")

---
## 5. Ensemble Sonuçları Karşılaştırması

In [ ]:
# Tum yontemlerin AUC matrisini olustur
all_methods = {}

# Bireysel modeller
for name in MODEL_NAMES:
    all_methods[name] = individual_aucs[name]

# Ensemble yontemleri
all_methods['Simple Average'] = simple_avg_aucs
all_methods['Weighted Average'] = weighted_avg_aucs
all_methods['Maximum'] = max_aucs
all_methods['Rank Average'] = rank_avg_aucs
all_methods['Optimal Weighted'] = optimal_aucs

# DataFrame olustur
full_auc_matrix = pd.DataFrame(all_methods).reindex(COMMON_14)
full_auc_matrix.index.name = 'Patoloji'

# Ortalama satiri ekle
mean_row = full_auc_matrix.mean()
full_auc_matrix.loc['ORTALAMA'] = mean_row

print("=" * 120)
print("Tam AUC-ROC Karsilastirma Matrisi")
print("=" * 120)
print(full_auc_matrix.round(4).to_string())

# CSV olarak kaydet
full_auc_matrix.to_csv(OUTPUT_DIR / 'cxr_ensemble_auc_comparison.csv')
print(f"\nKaydedildi: cxr_ensemble_auc_comparison.csv")

In [ ]:
# Ortalama AUC karsilastirma bar chart
method_names = list(all_methods.keys())
mean_aucs = []
colors = []

for method in method_names:
    valid = [v for v in all_methods[method].values() if not np.isnan(v)]
    mean_aucs.append(np.mean(valid))
    if method in MODEL_COLORS:
        colors.append(MODEL_COLORS[method])
    elif method in ENSEMBLE_COLORS:
        colors.append(ENSEMBLE_COLORS[method])
    else:
        colors.append('#95a5a6')

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(method_names)), mean_aucs, color=colors, edgecolor='white', linewidth=0.5)

# Deger etiketleri
for bar_obj, val in zip(bars, mean_aucs):
    ax.text(bar_obj.get_x() + bar_obj.get_width()/2, bar_obj.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Ortalama AUC-ROC')
ax.set_title('Yontem Bazinda Ortalama AUC-ROC Karsilastirmasi\n(14 NIH Patolojisi)', fontweight='bold')
ax.set_ylim(min(mean_aucs) - 0.02, max(mean_aucs) + 0.015)

# En iyi bireysel ve en iyi ensemble cizgileri
best_individual = max(mean_aucs[:3])
ax.axhline(y=best_individual, color='gray', linestyle='--', alpha=0.5,
           label=f'En iyi bireysel: {best_individual:.4f}')
ax.legend(loc='lower right')

# Bireysel vs Ensemble ayirici cizgi
ax.axvline(x=2.5, color='black', linestyle=':', alpha=0.3)
ax.text(1.0, max(mean_aucs) + 0.01, 'Bireysel', ha='center', fontsize=9, fontstyle='italic', color='gray')
ax.text(5.5, max(mean_aucs) + 0.01, 'Ensemble', ha='center', fontsize=9, fontstyle='italic', color='gray')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_mean_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_mean_auc_comparison.png")

In [ ]:
# Iyilestirme analizi: ensemble vs en iyi bireysel model
print("=" * 80)
print("Ensemble Iyilestirme Analizi (vs En Iyi Bireysel Model)")
print("=" * 80)

ensemble_methods_only = ['Simple Average', 'Weighted Average', 'Maximum', 'Rank Average', 'Optimal Weighted']

for ens_name in ensemble_methods_only:
    improvements = []
    better_count = 0
    worse_count = 0
    
    for p in COMMON_14:
        ens_auc = all_methods[ens_name][p]
        if np.isnan(ens_auc):
            continue
        best_ind = max(individual_aucs[m][p] for m in MODEL_NAMES if not np.isnan(individual_aucs[m][p]))
        diff = ens_auc - best_ind
        improvements.append(diff)
        if diff > 0.001:
            better_count += 1
        elif diff < -0.001:
            worse_count += 1
    
    avg_imp = np.mean(improvements)
    print(f"\n{ens_name}:")
    print(f"  Ortalama iyilestirme: {avg_imp:+.4f}")
    print(f"  Daha iyi: {better_count}/14 patoloji")
    print(f"  Daha kotu: {worse_count}/14 patoloji")
    print(f"  Esit: {14 - better_count - worse_count}/14 patoloji")

---
## 6. Patoloji Bazında Derinlemesine Analiz

In [ ]:
# Her patoloji icin en iyi yontem
print("=" * 80)
print("Patoloji Bazinda En Iyi Yontem")
print("=" * 80)

all_method_names = MODEL_NAMES + ensemble_methods_only

best_method_per_pathology = {}
for p in COMMON_14:
    best_method = None
    best_auc = -1
    for method in all_method_names:
        auc_val = all_methods[method][p]
        if not np.isnan(auc_val) and auc_val > best_auc:
            best_auc = auc_val
            best_method = method
    best_method_per_pathology[p] = best_method
    is_ensemble = best_method in ensemble_methods_only
    tag = "[ENSEMBLE]" if is_ensemble else "[BIREYSEL]"
    marker = " << AlpCAN" if p in ('Nodule', 'Mass') else ""
    print(f"  {p:>20s}: {best_method:>20s} (AUC={best_auc:.4f}) {tag}{marker}")

# Ozet
ensemble_wins = sum(1 for m in best_method_per_pathology.values() if m in ensemble_methods_only)
individual_wins = 14 - ensemble_wins
print(f"\nOzet: Ensemble {ensemble_wins}/14, Bireysel {individual_wins}/14 patolojide en iyi")

In [ ]:
# Nodule ve Mass icin ozel analiz (AlpCAN icin kritik)
print("=" * 80)
print("AlpCAN Kritik Patolojiler: Nodule ve Mass Detayli Analiz")
print("=" * 80)

for target_p in ['Nodule', 'Mass']:
    print(f"\n--- {target_p} ---")
    y_true = aligned_gt[target_p].values
    n_pos = int(y_true.sum())
    prevalence = n_pos / N_IMAGES * 100
    print(f"  Pozitif: {n_pos:,} / {N_IMAGES:,} ({prevalence:.2f}%)")
    
    print(f"\n  {'Yontem':>25s} | {'AUC':>8s} | {'Fark (vs Ark+)':>14s}")
    print(f"  {'-'*55}")
    
    ark_auc = individual_aucs['Ark+'][target_p]
    
    for method in all_method_names:
        auc_val = all_methods[method][target_p]
        if not np.isnan(auc_val):
            diff = auc_val - ark_auc
            marker = " <-- EN IYI" if method == best_method_per_pathology[target_p] else ""
            print(f"  {method:>25s} | {auc_val:>8.4f} | {diff:>+13.4f}{marker}")
    
    # Optimal agirliklar
    if target_p in optimal_weights_per_pathology:
        print(f"\n  Optimal agirliklar: {optimal_weights_per_pathology[target_p]}")

In [ ]:
# Model tahminleri arasi korelasyon analizi
print("Model Tahminleri Arasi Korelasyon Analizi")
print("(Modeller ne kadar tamamlayici?)")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Secilmis patolojiler icin korelasyon
focus_pathologies = ['Nodule', 'Mass', 'Atelectasis', 'Cardiomegaly', 'Effusion', 'Pneumonia']

for idx, p in enumerate(focus_pathologies):
    row_idx = idx // 3
    col_idx = idx % 3
    ax = axes[row_idx, col_idx]
    
    # 3 model tahminlerinden korelasyon matrisi
    pred_data = pd.DataFrame({
        name: get_pred_array(name, p) for name in MODEL_NAMES
    })
    corr = pred_data.corr()
    
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
                vmin=0.3, vmax=1.0, ax=ax, square=True,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{p}', fontweight='bold')

plt.suptitle('Model Tahminleri Arasi Pearson Korelasyonu\n(Dusuk korelasyon = daha tamamlayici modeller)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_correlation.png")

# Genel korelasyon ozeti
print(f"\nGenel Korelasyon Ozeti (tum patolojiler ortalamasi):")
all_corrs = {}
for i, m1 in enumerate(MODEL_NAMES):
    for j, m2 in enumerate(MODEL_NAMES):
        if i < j:
            pair_corrs = []
            for p in COMMON_14:
                c = np.corrcoef(get_pred_array(m1, p), get_pred_array(m2, p))[0, 1]
                pair_corrs.append(c)
            mean_corr = np.mean(pair_corrs)
            all_corrs[f"{m1} vs {m2}"] = mean_corr
            print(f"  {m1} vs {m2}: {mean_corr:.4f}")

print(f"\n  Not: Dusuk korelasyon ensemble icin daha faydalıdir.")

---
## 7. Güven Analizi (Tahmin Uyumu)

In [ ]:
# Tahmin uyumu analizi: 3 model ne zaman hemfikir?
# Threshold: 0.5 (standart sigmoid threshold)
AGREEMENT_THRESHOLD = 0.5

print("=" * 80)
print(f"Tahmin Uyumu Analizi (threshold = {AGREEMENT_THRESHOLD})")
print("=" * 80)

agreement_stats = []

for p in COMMON_14:
    y_true = aligned_gt[p].values
    
    # Her modelin binary tahminleri
    binary_preds = np.zeros((N_IMAGES, len(MODEL_NAMES)))
    for j, name in enumerate(MODEL_NAMES):
        binary_preds[:, j] = (get_pred_array(name, p) >= AGREEMENT_THRESHOLD).astype(int)
    
    # Oy sayisi
    vote_sum = binary_preds.sum(axis=1).astype(int)  # 0, 1, 2, veya 3
    
    # Tam uyum: 3/3 pozitif veya 0/3 (hepsi negatif)
    all_agree_pos = (vote_sum == 3).sum()
    all_agree_neg = (vote_sum == 0).sum()
    total_agree = all_agree_pos + all_agree_neg
    agree_pct = total_agree / N_IMAGES * 100
    
    # Kismi uyum
    partial_2 = (vote_sum == 2).sum()  # 2/3 pozitif
    partial_1 = (vote_sum == 1).sum()  # 1/3 pozitif
    
    agreement_stats.append({
        'pathology': p,
        'all_agree_pos': all_agree_pos,
        'all_agree_neg': all_agree_neg,
        'vote_2_of_3': partial_2,
        'vote_1_of_3': partial_1,
        'agreement_pct': agree_pct,
        'n_positive': int(y_true.sum()),
    })
    
    marker = " << AlpCAN" if p in ('Nodule', 'Mass') else ""
    print(f"  {p:>20s}: Tam uyum {agree_pct:>5.1f}% | "
          f"3/3:{all_agree_pos:>6,} | 2/3:{partial_2:>6,} | "
          f"1/3:{partial_1:>6,} | 0/3:{all_agree_neg:>6,}{marker}")

agreement_df = pd.DataFrame(agreement_stats)
print(f"\nOrtalama tam uyum orani: {agreement_df['agreement_pct'].mean():.1f}%")

In [ ]:
# Yuksek guvenli vakalar analizi
# Tum modeller hemfikir VE olasilik > 0.7
HIGH_CONF_THRESHOLD = 0.7

print("=" * 80)
print(f"Yuksek Guven Analizi (tum modeller > {HIGH_CONF_THRESHOLD})")
print("=" * 80)

high_conf_stats = []

for p in COMMON_14:
    y_true = aligned_gt[p].values
    all_p = get_all_preds(p)  # (N, 3)
    
    # Tum modeller > threshold
    all_high = (all_p > HIGH_CONF_THRESHOLD).all(axis=1)
    # Tum modeller < (1 - threshold)
    all_low = (all_p < (1 - HIGH_CONF_THRESHOLD)).all(axis=1)
    
    n_high_conf_pos = all_high.sum()
    n_high_conf_neg = all_low.sum()
    
    # Yuksek guvenli pozitif tahminlerin dogrulugu
    if n_high_conf_pos > 0:
        precision_high = y_true[all_high].mean()
    else:
        precision_high = np.nan
    
    if n_high_conf_neg > 0:
        npv_high = (1 - y_true[all_low]).mean()
    else:
        npv_high = np.nan
    
    high_conf_stats.append({
        'pathology': p,
        'n_high_pos': n_high_conf_pos,
        'n_high_neg': n_high_conf_neg,
        'precision': precision_high,
        'npv': npv_high,
    })
    
    prec_str = f"{precision_high:.3f}" if not np.isnan(precision_high) else "N/A"
    npv_str = f"{npv_high:.3f}" if not np.isnan(npv_high) else "N/A"
    
    print(f"  {p:>20s}: Yuksek-poz={n_high_conf_pos:>5,} (prec={prec_str}) | "
          f"Yuksek-neg={n_high_conf_neg:>6,} (NPV={npv_str})")

high_conf_df = pd.DataFrame(high_conf_stats)

In [ ]:
# Uyumsuzluk analizi ve gorsellistirme
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Uyum oranları bar chart
ax = axes[0]
pathology_order = agreement_df.sort_values('agreement_pct', ascending=True)['pathology'].values
agree_vals = agreement_df.set_index('pathology').loc[pathology_order, 'agreement_pct'].values
bar_colors = ['#e74c3c' if p in ('Nodule', 'Mass') else '#3498db' for p in pathology_order]
ax.barh(range(len(pathology_order)), agree_vals, color=bar_colors)
ax.set_yticks(range(len(pathology_order)))
ax.set_yticklabels(pathology_order)
ax.set_xlabel('Tam Uyum Orani (%)')
ax.set_title('Model Uyum Oranlari', fontweight='bold')
ax.axvline(x=90, color='gray', linestyle='--', alpha=0.5)

# 2. Oylama dagilimi — Nodule ve Mass
ax = axes[1]
for target_p in ['Nodule', 'Mass']:
    binary_preds_t = np.zeros((N_IMAGES, len(MODEL_NAMES)))
    for j, name in enumerate(MODEL_NAMES):
        binary_preds_t[:, j] = (get_pred_array(name, target_p) >= AGREEMENT_THRESHOLD).astype(int)
    vote_sum_t = binary_preds_t.sum(axis=1).astype(int)
    
    # Gercek pozitifler icinde oylama dagilimi
    y_true_t = aligned_gt[target_p].values
    pos_mask = y_true_t == 1
    if pos_mask.sum() > 0:
        pos_votes = vote_sum_t[pos_mask]
        vote_dist = [100.0 * (pos_votes == v).sum() / len(pos_votes) for v in range(4)]
        x_pos = np.arange(4)
        offset = -0.15 if target_p == 'Nodule' else 0.15
        color = '#e74c3c' if target_p == 'Nodule' else '#e67e22'
        ax.bar(x_pos + offset, vote_dist, 0.3, label=f'{target_p} pozitif', color=color)

ax.set_xticks(range(4))
ax.set_xticklabels(['0/3', '1/3', '2/3', '3/3'])
ax.set_xlabel('Pozitif oy sayisi')
ax.set_ylabel('Gercek pozitiflerin yuzdesi (%)')
ax.set_title('Gercek Pozitiflerde Oylama\n(Nodule & Mass)', fontweight='bold')
ax.legend()

# 3. Uyum vs AUC iliskisi
ax = axes[2]
for p in COMMON_14:
    agree_pct = agreement_df[agreement_df['pathology'] == p]['agreement_pct'].values[0]
    # En iyi ensemble AUC
    best_ens_auc = max(all_methods[m][p] for m in ensemble_methods_only if not np.isnan(all_methods[m][p]))
    color = '#e74c3c' if p in ('Nodule', 'Mass') else '#3498db'
    size = 120 if p in ('Nodule', 'Mass') else 60
    ax.scatter(agree_pct, best_ens_auc, c=color, s=size, zorder=3)
    ax.annotate(p, (agree_pct, best_ens_auc), fontsize=7,
                xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('Tam Uyum Orani (%)')
ax.set_ylabel('En Iyi Ensemble AUC')
ax.set_title('Uyum Orani vs Ensemble Performansi', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_agreement.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_agreement.png")

---
## 8. AlpCAN Pipeline Entegrasyonu

In [ ]:
# AlpCAN icin onerilen ensemble konfigurasyonu
# En iyi genel performansi veren yontemi sec

# Her yontem icin ortalama AUC hesapla
method_summary = {}
for method in all_method_names:
    valid = [v for v in all_methods[method].values() if not np.isnan(v)]
    method_summary[method] = {
        'mean_auc': np.mean(valid),
        'nodule_auc': all_methods[method].get('Nodule', np.nan),
        'mass_auc': all_methods[method].get('Mass', np.nan),
    }

# En iyi genel yontem (Optimal haric — o oracle)
non_oracle_methods = [m for m in all_method_names if m != 'Optimal Weighted']
best_general = max(non_oracle_methods, key=lambda m: method_summary[m]['mean_auc'])

print("=" * 80)
print("AlpCAN CXR Pipeline — Onerilen Ensemble Konfigurasyonu")
print("=" * 80)

print(f"\nEn iyi genel yontem (oracle haric): {best_general}")
print(f"  Ortalama AUC: {method_summary[best_general]['mean_auc']:.4f}")
print(f"  Nodule AUC: {method_summary[best_general]['nodule_auc']:.4f}")
print(f"  Mass AUC: {method_summary[best_general]['mass_auc']:.4f}")

# Final ensemble tahminlerini olustur
# Strateji: Rank Average (genellikle en robust) veya best_general
# Burada en iyi genel yontemi kullanalim

# Her ensemble yonteminin tahminlerini bir dict'te tut
ensemble_pred_dicts = {
    'Simple Average': simple_avg_preds,
    'Weighted Average': weighted_avg_preds,
    'Maximum': max_preds,
    'Rank Average': rank_avg_preds,
    'Optimal Weighted': optimal_preds,
}

# Secilen yontem ile final tahminleri olustur
final_method = best_general
final_preds_dict = ensemble_pred_dicts.get(final_method, None)

# Eger bireysel model en iyiyse, weighted average kullan
if final_preds_dict is None:
    print(f"\n  Not: En iyi yontem bireysel model ({final_method}).")
    print(f"  AlpCAN pipeline icin yine de ensemble (Weighted Average) onerilir.")
    final_method = 'Weighted Average'
    final_preds_dict = weighted_avg_preds

print(f"\nKullanilan final yontem: {final_method}")

In [ ]:
# Ensemble tahminlerini CSV olarak kaydet
ensemble_results_df = pd.DataFrame({'image_name': aligned_preds[MODEL_NAMES[0]]['image_name'].values})

for p in COMMON_14:
    ensemble_results_df[f'pred_{p}'] = final_preds_dict[p]
    ensemble_results_df[f'gt_{p}'] = aligned_gt[p].values

ensemble_results_df.to_csv(OUTPUT_DIR / 'cxr_ensemble_predictions.csv', index=False)
print(f"Kaydedildi: cxr_ensemble_predictions.csv")
print(f"  Satir: {len(ensemble_results_df):,}")
print(f"  Sutun: {ensemble_results_df.shape[1]}")
print(f"  Yontem: {final_method}")

# Optimal agirliklari JSON olarak kaydet
ensemble_config = {
    'ensemble_method': final_method,
    'models': MODEL_NAMES,
    'n_images': N_IMAGES,
    'pathologies': COMMON_14,
    'global_weights': {name: round(w, 4) for name, w in global_weights.items()},
    'optimal_weights_per_pathology': optimal_weights_per_pathology,
    'model_mean_aucs': {name: round(v, 4) for name, v in model_mean_aucs.items()},
    'ensemble_mean_auc': round(method_summary[final_method]['mean_auc'], 4),
}

with open(OUTPUT_DIR / 'cxr_ensemble_weights.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)
print(f"\nKaydedildi: cxr_ensemble_weights.json")

# Ozet metrikleri CSV olarak kaydet
summary_rows = []
for method in all_method_names:
    valid = [v for v in all_methods[method].values() if not np.isnan(v)]
    summary_rows.append({
        'method': method,
        'type': 'individual' if method in MODEL_NAMES else 'ensemble',
        'mean_auc': np.mean(valid),
        'nodule_auc': all_methods[method].get('Nodule', np.nan),
        'mass_auc': all_methods[method].get('Mass', np.nan),
        'nodule_mass_mean': np.nanmean([all_methods[method].get('Nodule', np.nan),
                                         all_methods[method].get('Mass', np.nan)]),
        'min_auc': min(valid),
        'max_auc': max(valid),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('mean_auc', ascending=False)
summary_df.to_csv(OUTPUT_DIR / 'cxr_ensemble_summary.csv', index=False)
print(f"Kaydedildi: cxr_ensemble_summary.csv")
print()
print(summary_df.round(4).to_string(index=False))

---
## 9. Görselleştirme Özeti

In [ ]:
# ROC egrileri karsilastirmasi — anahtar patolojiler
key_pathologies = ['Nodule', 'Mass', 'Atelectasis', 'Cardiomegaly']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, target_p in enumerate(key_pathologies):
    row_idx = idx // 2
    col_idx = idx % 2
    ax = axes[row_idx, col_idx]
    
    y_true = aligned_gt[target_p].values
    n_pos = int(y_true.sum())
    
    if n_pos == 0 or n_pos == N_IMAGES:
        ax.text(0.5, 0.5, 'Veri yetersiz', ha='center', va='center')
        continue
    
    # Bireysel modeller
    for name in MODEL_NAMES:
        y_score = get_pred_array(name, target_p)
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc_val = individual_aucs[name][target_p]
        ax.plot(fpr, tpr, label=f'{name} ({auc_val:.3f})',
                color=MODEL_COLORS[name], linewidth=1.5, alpha=0.7)
    
    # En iyi ensemble (oracle haric)
    best_ens_name = max(
        [m for m in ensemble_methods_only if m != 'Optimal Weighted'],
        key=lambda m: all_methods[m][target_p] if not np.isnan(all_methods[m][target_p]) else -1
    )
    best_ens_pred = ensemble_pred_dicts[best_ens_name][target_p]
    fpr_e, tpr_e, _ = roc_curve(y_true, best_ens_pred)
    ens_auc = all_methods[best_ens_name][target_p]
    ax.plot(fpr_e, tpr_e, label=f'{best_ens_name} ({ens_auc:.3f})',
            color=ENSEMBLE_COLORS[best_ens_name], linewidth=2.5, linestyle='--')
    
    # Optimal (oracle)
    if target_p in optimal_preds:
        fpr_o, tpr_o, _ = roc_curve(y_true, optimal_preds[target_p])
        opt_auc = optimal_aucs[target_p]
        ax.plot(fpr_o, tpr_o, label=f'Optimal ({opt_auc:.3f})',
                color=ENSEMBLE_COLORS['Optimal Weighted'], linewidth=2, linestyle=':')
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.2)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{target_p} (N+={n_pos:,})', fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('ROC Egrileri Karsilastirmasi — Bireysel Modeller vs Ensemble',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_roc_curves.png")

In [ ]:
# Radar (spider) chart — tum 14 patoloji AUC karsilastirmasi
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Acilar
angles = np.linspace(0, 2 * np.pi, len(COMMON_14), endpoint=False).tolist()
angles += angles[:1]  # Kapama noktasi

# Cizilecek yontemler
radar_methods = MODEL_NAMES + ['Simple Average', 'Weighted Average', 'Optimal Weighted']
radar_colors = {
    'TorchXRayVision': ('#3498db', '-', 1.2),
    'ResNet-50': ('#2ecc71', '-', 1.2),
    'Ark+': ('#e74c3c', '-', 1.5),
    'Simple Average': ('#9b59b6', '--', 2.0),
    'Weighted Average': ('#e67e22', '--', 2.0),
    'Optimal Weighted': ('#f1c40f', ':', 2.0),
}

for method in radar_methods:
    values = [all_methods[method][p] if not np.isnan(all_methods[method][p]) else 0.5 for p in COMMON_14]
    values += values[:1]  # Kapama noktasi
    color, ls, lw = radar_colors[method]
    ax.plot(angles, values, label=method, color=color, linestyle=ls, linewidth=lw)
    ax.fill(angles, values, color=color, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(COMMON_14, fontsize=8)
ax.set_ylim(0.55, 1.0)
ax.set_rticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_rlabel_position(30)
ax.set_title('14 Patoloji AUC-ROC Radar Grafigi\nBireysel Modeller vs Ensemble Yontemleri',
             fontweight='bold', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_radar.png")

In [ ]:
# Tam AUC heatmap: tum modeller + tum ensemble yontemleri
fig, ax = plt.subplots(figsize=(14, 10))

# ORTALAMA satirini cikar
heatmap_full = full_auc_matrix.drop('ORTALAMA', errors='ignore')

sns.heatmap(
    heatmap_full,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    vmin=0.60, vmax=1.0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'AUC-ROC'}
)

ax.set_title('AUC-ROC Tam Karsilastirma Isitma Haritasi\n'
             '(3 Bireysel Model + 5 Ensemble Yontemi x 14 Patoloji)',
             fontweight='bold')
ax.set_ylabel('Patoloji')
ax.set_xlabel('Yontem')

# Nodule ve Mass vurgula
for i, p in enumerate(COMMON_14):
    if p in ('Nodule', 'Mass'):
        ax.add_patch(plt.Rectangle((0, i), heatmap_full.shape[1], 1,
                                    fill=False, edgecolor='red', linewidth=2.5))

# Bireysel / Ensemble ayirici
ax.axvline(x=3, color='black', linewidth=2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kaydedildi: cxr_ensemble_heatmap.png")

---
## Final Özet

In [ ]:
# Cikti dosyalarini listele
print("=" * 80)
print("Kaydedilen Dosyalar")
print("=" * 80)

output_files = sorted(OUTPUT_DIR.glob('cxr_ensemble_*'))
for f in output_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    size_str = f"{size_mb:.1f} MB" if size_mb >= 1 else f"{f.stat().st_size / 1024:.1f} KB"
    print(f"  {f.name} ({size_str})")

print(f"\nToplam: {len(output_files)} dosya")

In [ ]:
# ==========================================
# KAPSAMLI FINAL OZET
# ==========================================

print("\n" + "=" * 80)
print("AlpCAN CXR Multi-Model Ensemble Pipeline — FINAL OZET")
print("=" * 80)

print(f"\n--- Veri Seti ---")
print(f"  NIH ChestX-ray14")
print(f"  Ortak goruntu sayisi: {N_IMAGES:,}")
print(f"  Patoloji sayisi: {len(COMMON_14)}")

print(f"\n--- Modeller ---")
for name in MODEL_NAMES:
    mean_auc = method_summary[name]['mean_auc']
    print(f"  {name:>20s}: Ortalama AUC = {mean_auc:.4f}")

print(f"\n--- Ensemble Sonuclari ---")
for method in ensemble_methods_only:
    mean_auc = method_summary[method]['mean_auc']
    best_ind = max(method_summary[m]['mean_auc'] for m in MODEL_NAMES)
    diff = mean_auc - best_ind
    print(f"  {method:>20s}: Ortalama AUC = {mean_auc:.4f} (vs en iyi bireysel: {diff:+.4f})")

print(f"\n--- AlpCAN Kritik Patolojiler ---")
for target_p in ['Nodule', 'Mass']:
    print(f"  {target_p}:")
    for method in all_method_names:
        auc_val = all_methods[method][target_p]
        if not np.isnan(auc_val):
            marker = " <- en iyi" if method == best_method_per_pathology[target_p] else ""
            print(f"    {method:>25s}: {auc_val:.4f}{marker}")

print(f"\n--- Onerilen Konfiguerasyon ---")
print(f"  Yontem: {final_method}")
print(f"  Ortalama AUC: {method_summary[final_method]['mean_auc']:.4f}")
if final_method == 'Weighted Average':
    print(f"  Agirliklar:")
    for name, w in global_weights.items():
        print(f"    {name}: {w:.4f}")

print(f"\n--- Model Tamamlayicilik ---")
for pair, corr_val in all_corrs.items():
    complementary = "Yuksek tamamlayicilik" if corr_val < 0.6 else \
                    "Orta tamamlayicilik" if corr_val < 0.8 else \
                    "Dusuk tamamlayicilik"
    print(f"  {pair}: r={corr_val:.4f} ({complementary})")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Cross-validation ile optimal agirliklari dogrula")
print(f"  2. Ensemble tahminlerini CXR Pipeline Agent'a entegre et")
print(f"  3. CT Pipeline sonuclari ile CXR ensemble'i birlestir")
print(f"  4. AlpCAN karar destek sistemi threshold optimizasyonu")
print(f"  5. Klinik senaryo bazli performans degerlendirmesi")
print("=" * 80)

---

## Sonuç ve Değerlendirme

Bu notebook'ta 3 farklı CXR modelinin (TorchXRayVision DenseNet-121, ResNet-50, Ark+ Swin-L) tahminleri 5 farklı ensemble yöntemi ile birleştirildi:

1. **Basit Ortalama** — Eşit ağırlıklı aritmetik ortalama
2. **Ağırlıklı Ortalama** — Model AUC'sine orantılı ağırlıklar
3. **Maksimum** — En yüksek olasılık değeri
4. **Rank Ortalaması** — Percentile dönüştürülmüş ortalama
5. **Optimal Ağırlıklı (Oracle)** — Patoloji başına grid search

### Temel Bulgular:
- **Ark+ Swin-L** bireysel modeller içinde en yüksek performansa sahip
- **Ensemble yöntemleri** bireysel modellere kıyasla tutarlı iyileştirme sağlıyor
- **Modellerin tamamlayıcılığı** ensemble performansını belirleyen en önemli faktör
- **Nodule ve Mass** patolojileri AlpCAN için kritik — ensemble bu patolojilerde de iyileştirme sağlıyor

### AlpCAN Pipeline İçin Öneri:
- **Ağırlıklı ortalama** veya **rank ortalaması** en robust ensemble yöntemleri
- Optimal ağırlıklar cross-validation ile doğrulanmalı
- Yüksek güvenli vakalar (tüm modeller hemfikir) klinik karar desteği için kullanılabilir